# 🛰️ Prospectividad por alteración hidrotermal con Sentinel-2

**Cuaderno para Google Colab** — de la imagen a las zonas de interés, en 4 pasos.

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jhon21geo/Teledeteccion/blob/main/prospectividad_sentinel2.ipynb)

**Enlace directo:** https://colab.research.google.com/github/jhon21geo/Teledeteccion/blob/main/prospectividad_sentinel2.ipynb

| Paso | Qué hace |
|---|---|
| 1. Datos | Imagen de ejemplo de este repo, **descarga** Sentinel-2 L2A de tu zona (gratis, sin cuenta) o **sube** tu GeoTIFF |
| 2. Análisis espectral | **M1** índices de bandas · **M2** PCA dirigido (Crosta) · **M3** ángulo espectral (SAM) → consenso ≥ 2 de 3 |
| 3. Lineamientos | Canny + Hough; opcional con DEM Copernicus 30 m y diagrama de rosa |
| 4. Zonas | Alteración + densidad de lineamientos → GeoJSON, KMZ, CSV y GeoTIFF |

> **Cómo usarlo:** abre el badge de Colab → *Entorno de ejecución → Ejecutar todo*. Para tu zona cambia solo la celda **Parámetros**.
>
> Este cuaderno **no está pensado para correr en tu PC**. Colab instala las librerías y clona los datos.
>
> ⚠️ **Alcance:** Sentinel-2 distingue *arcillas/hidroxilos* (Al-OH, 2.2 µm) y *óxidos de Fe*; **no** separa argílica avanzada, fílica ni propilítica (eso pide SWIR fino: ASTER, WorldView-3, EMIT, EnMAP, PRISMA). Es **priorización**, no reemplaza el mapeo ni la validación de campo.


## 0 · Colab: instalación, datos del repo e importaciones


In [ ]:
import sys, subprocess, os, json, zipfile, warnings, urllib.request
warnings.filterwarnings('ignore')

COLAB_URL = 'https://colab.research.google.com/github/jhon21geo/Teledeteccion/blob/main/prospectividad_sentinel2.ipynb'
REPO_URL = 'https://github.com/jhon21geo/Teledeteccion.git'
ROOT = '/content/Teledeteccion'

IN_COLAB = 'google.colab' in sys.modules
if not IN_COLAB:
    raise SystemExit(
        'Este cuaderno solo se ejecuta en Google Colab.\n'
        'Ábrelo aquí:\n' + COLAB_URL
    )

def _pip(*pkgs):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs], check=True)

# rasterio y pystac no vienen en Colab; sklearn / skimage / scipy sí.
_pip('rasterio', 'pystac-client')

tif = os.path.join(ROOT, 'data', 'prospectividad_sentinel2.tif')
if not os.path.isfile(tif):
    print('Clonando jhon21geo/Teledeteccion (imagen de ejemplo)…')
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, ROOT], check=True)
os.chdir(ROOT)

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import rasterio
from rasterio import features, windows
from rasterio.transform import from_bounds
from rasterio.warp import transform_bounds, transform_geom, reproject, Resampling
from scipy import ndimage as ndi
from sklearn.decomposition import PCA
from skimage.feature import canny
from skimage.transform import probabilistic_hough_line
from skimage.draw import line as draw_line
from skimage.filters import gaussian

plt.rcParams.update({'figure.dpi': 110, 'axes.titlesize': 11, 'axes.titleweight': 'bold'})
OUT = 'resultados'
os.makedirs(OUT, exist_ok=True)
print('Colab listo · rasterio', rasterio.__version__, '· carpeta', os.getcwd())


## 1 · Parámetros
Todo lo que normalmente querrás cambiar está aquí.

In [ ]:
# ───────────── 1. DATOS ─────────────
MODO = 'ejemplo'          # 'ejemplo' | 'descargar' | 'subir' | 'drive'

# — si MODO = 'descargar' (Sentinel-2 L2A, AWS Earth Search, sin cuenta) —
LON, LAT   = -72.0, -14.0   # centro del área (grados decimales)  ← cámbialo
LADO_KM    = 10             # lado del cuadrado a descargar (km)
FECHAS     = '2024-05-01/2024-09-30'   # época seca = menos nubes
NUBES_MAX  = 15             # % de nubes máximo de la escena
# — si MODO = 'subir' — selector de Colab (mejor < ~50 MB)
# — si MODO = 'drive' — imagen grande (~200 MB): súbela a Drive —
RUTA_DRIVE = '/content/drive/MyDrive/teledeteccion/mi_escena.tif'
# GeoTIFF 6 bandas B2, B3, B4, B8, B11, B12 (reflectancia 0–1 o entero ×10 000)

# ───────────── 2. ANÁLISIS ESPECTRAL ─────────────
NDVI_MAX   = 0.40           # por encima = vegetación (se enmascara)
PERCENTIL  = 92             # umbral de cada método: se marca el 8 % más "alterado" de la escena
SEMILLA_PCT = 2             # M3: % de píxeles M1 más extremos usados como firma de referencia
PUNTOS_REF = {}             # M3 con tus propias firmas, ej. {'argilica': [(lon, lat), ...], 'oxidos': [...]}

# ───────────── 3. LINEAMIENTOS ─────────────
USAR_DEM   = False          # True = descarga DEM Copernicus 30 m y extrae lineamientos del relieve (recomendado)
SIGMA_LIN  = 2.0            # suavizado antes de detectar bordes (px)
Q_BAJO, Q_ALTO = 0.85, 0.95  # sensibilidad: cuantiles de gradiente (sube para menos lineamientos, baja para más)
LONG_MIN_M = 300            # largo mínimo de un lineamiento (m)
MAX_LIN    = 600            # se conservan los N más largos

# ───────────── 4. ZONAS ─────────────
PESO_ALTERACION = 0.6       # peso de la alteración (0–1); el resto va a lineamientos
PERCENTIL_ALTA  = 95        # zona "Alta" = 5 % superior del puntaje
AREA_MIN_KM2    = 0.05      # se descartan zonas más pequeñas
SUAVIZADO_M     = 150       # radio de influencia para convertir alteración en densidad (m)


## 2 · Datos
La escena se estandariza a un arreglo `R` de **6 bandas × alto × ancho** en reflectancia (0–1): `B2 azul · B3 verde · B4 rojo · B8 NIR · B11 SWIR1 · B12 SWIR2`.

Un GeoTIFF de **~200 MB** no cabe en GitHub. Usa `MODO = 'drive'` y Google Drive.


In [ ]:
BANDAS = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12']
REPO_RAW = 'https://raw.githubusercontent.com/jhon21geo/Teledeteccion/main/data/'
EJEMPLO = 'prospectividad_sentinel2.tif'


def _normaliza(a):
    """Convierte a reflectancia 0–1 si viene como entero ×10 000."""
    a = a.astype('float32')
    if np.nanpercentile(a[a > 0], 99) > 2:
        a = a / 10000.0
        print('  · detectado entero ×10 000 → reflectancia 0–1')
    return a


def leer_geotiff(ruta):
    with rasterio.open(ruta) as src:
        if src.count < 6:
            raise ValueError(f'El GeoTIFF tiene {src.count} bandas; se necesitan 6 (B2, B3, B4, B8, B11, B12).')
        if src.crs is None:
            raise ValueError('El GeoTIFF no trae sistema de coordenadas (CRS).')
        a = src.read(list(range(1, 7)))
        return dict(R=_normaliza(a), transform=src.transform, crs=src.crs, nombre=os.path.basename(ruta))


def descargar_s2(lon, lat, lado_km, fechas, nubes_max):
    """Mejor escena Sentinel-2 L2A (menos nubes) recortada alrededor de (lon, lat). Sin cuenta ni claves."""
    from pystac_client import Client
    cat = Client.open('https://earth-search.aws.element84.com/v1')
    d = lado_km / 2 / 111.32
    bbox = [lon - d / np.cos(np.radians(lat)), lat - d, lon + d / np.cos(np.radians(lat)), lat + d]
    items = cat.search(collections=['sentinel-2-l2a'], bbox=bbox, datetime=fechas,
                       query={'eo:cloud_cover': {'lt': nubes_max}}, max_items=100).item_collection()
    if len(items) == 0:
        raise RuntimeError('No hay escenas con ese criterio: amplía FECHAS o sube NUBES_MAX.')
    items = sorted(items, key=lambda it: it.properties.get('eo:cloud_cover', 100))
    it = items[0]
    print(f'  · {len(items)} escenas candidatas · elegida {it.id} · nubes {it.properties.get("eo:cloud_cover"):.1f} %')
    ASSET = {'B2': 'blue', 'B3': 'green', 'B4': 'red', 'B8': 'nir', 'B11': 'swir16', 'B12': 'swir22'}
    with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR', AWS_NO_SIGN_REQUEST='YES'):
        with rasterio.open(it.assets['red'].href) as ref:
            b = transform_bounds('EPSG:4326', ref.crs, *bbox)
            win = ref.window(*b).round_offsets().round_lengths()
            h, w = int(win.height), int(win.width)
            bnds = windows.bounds(win, ref.transform)
            tr, crs = from_bounds(*bnds, w, h), ref.crs
        capas = []
        for k in BANDAS:
            asset = it.assets[ASSET[k]]
            rb = (asset.extra_fields.get('raster:bands') or [{}])[0]
            esc, off = rb.get('scale', 1e-4), rb.get('offset', 0.0)
            with rasterio.open(asset.href) as s:
                dn = s.read(1, window=s.window(*bnds), out_shape=(h, w), resampling=Resampling.bilinear).astype('float32')
            ref_ = np.where(dn > 0, dn * esc + off, 0).astype('float32')
            capas.append(ref_)
        R = np.stack(capas)
        # máscara de calidad de la escena (SCL): sombras, nubes, cirros, agua y nieve → 0
        if 'scl' in it.assets:
            with rasterio.open(it.assets['scl'].href) as s:
                scl = s.read(1, window=s.window(*bnds), out_shape=(h, w), resampling=Resampling.nearest)
            R[:, np.isin(scl, [0, 1, 3, 8, 9, 10, 11])] = 0
    out = os.path.join(OUT, 'escena_descargada.tif')
    with rasterio.open(out, 'w', driver='GTiff', height=h, width=w, count=6, dtype='float32', crs=crs, transform=tr, compress='deflate') as dst:
        dst.write(R)
        dst.descriptions = tuple(BANDAS)
    print(f'  · guardada en {out} ({w}×{h} px)')
    return dict(R=R, transform=tr, crs=crs, nombre=it.id)


print(f'MODO = {MODO}')
if MODO == 'ejemplo':
    ruta = EJEMPLO if os.path.exists(EJEMPLO) else os.path.join('data', EJEMPLO)
    if not os.path.exists(ruta):
        ruta = os.path.join(OUT, EJEMPLO)
        print('  · descargando imagen de ejemplo desde GitHub…')
        urllib.request.urlretrieve(REPO_RAW + EJEMPLO, ruta)
    ESC = leer_geotiff(ruta)
elif MODO == 'descargar':
    ESC = descargar_s2(LON, LAT, LADO_KM, FECHAS, NUBES_MAX)
elif MODO == 'subir':
    try:
        from google.colab import files
        sub = files.upload()
        ESC = leer_geotiff(list(sub)[0])
    except ImportError:
        raise SystemExit('MODO "subir" usa el selector de archivos de Colab.')
elif MODO == 'drive':
    from google.colab import drive
    if not os.path.isdir('/content/drive/MyDrive'):
        drive.mount('/content/drive')
    if not os.path.isfile(RUTA_DRIVE):
        raise FileNotFoundError('No está el GeoTIFF. Súbelo a Drive y revisa RUTA_DRIVE: ' + RUTA_DRIVE)
    print('  · leyendo', RUTA_DRIVE)
    ESC = leer_geotiff(RUTA_DRIVE)
else:
    raise ValueError('MODO debe ser "ejemplo", "descargar", "subir" o "drive"')

R, TR, CRS = ESC['R'], ESC['transform'], ESC['crs']
NB, H, W = R.shape
PX = abs(TR.a)                      # tamaño de píxel (m)
KM2_PX = (TR.a * TR.e) and abs(TR.a * TR.e) / 1e6
print(f'Escena: {ESC["nombre"]} · {W}×{H} px · píxel {PX:.0f} m · área {W*H*KM2_PX:.1f} km² · CRS {CRS}')


In [ ]:
def rgb(R, p=(2, 98), gamma=0.8):
    c = np.stack([R[2], R[1], R[0]], -1)
    out = np.zeros_like(c)
    for i in range(3):
        v = c[..., i][c[..., i] > 0]
        lo, hi = np.percentile(v, p) if v.size else (0, 1)
        out[..., i] = np.clip((c[..., i] - lo) / (hi - lo + 1e-9), 0, 1) ** gamma
    return out

ext = (TR.c, TR.c + W * TR.a, TR.f + H * TR.e, TR.f)   # extensión en coordenadas del mapa (m)
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.imshow(rgb(R), extent=ext)
ax.set_title('Composición color natural (B4-B3-B2)')
ax.set_xlabel('Este (m)'); ax.set_ylabel('Norte (m)')
ax.ticklabel_format(style='plain'); plt.tight_layout(); plt.show()

## 3 · Análisis espectral
**Máscara previa:** se excluyen píxeles sin dato, **vegetación** (NDVI alto), **agua** y nubes/nieve brillantes, porque tapan la señal mineral.

### Los tres métodos
* **M1 — Índices de bandas.** *Arcillas / hidroxilos:* `B11 / B12` (la absorción Al-OH cerca de 2.2 µm baja B12 respecto a B11). *Óxidos de Fe:* `B4 / B2` (los óxidos absorben el azul y reflejan el rojo).
* **M2 — PCA dirigido (Crosta).** PCA sobre bandas elegidas; se escoge automáticamente la componente cuyos coeficientes tienen **signo opuesto** en las bandas que contrastan con el mineral buscado (así se aísla su firma y se reduce el efecto de brillo y topografía).
* **M3 — Ángulo espectral (SAM).** Ángulo entre el espectro de cada píxel y una firma de referencia; menor ángulo = más parecido. Firma: puntos que tú pasas en `PUNTOS_REF`, o, si no hay, **semilla automática** (píxeles más extremos de M1; ⚠️ esto hace M3 menos independiente de M1 — con firmas reales de campo mejora mucho).

**Consenso** = zona donde coinciden **≥ 2 de los 3** métodos.

In [ ]:
B = dict(zip(BANDAS, R))
eps = 1e-6
ndvi = (B['B8'] - B['B4']) / (B['B8'] + B['B4'] + eps)
mndwi = (B['B3'] - B['B11']) / (B['B3'] + B['B11'] + eps)
sin_dato = (R <= 0).any(0) | (R > 1.5).any(0)
MASCARA = (~sin_dato) & (ndvi < NDVI_MAX) & (mndwi < 0.1) & (B['B2'] < 0.30)
print(f'Píxeles analizables: {MASCARA.mean()*100:.1f} %  (excluidos: sin dato {sin_dato.mean()*100:.1f} %, vegetación {(ndvi>=NDVI_MAX).mean()*100:.1f} %)')

def pct_mask(x, pct, alto=True):
    """Marca el pct superior (o inferior) de x dentro de la máscara."""
    v = x[MASCARA]
    t = np.percentile(v, pct if alto else 100 - pct)
    m = (x >= t) if alto else (x <= t)
    return m & MASCARA

# ── M1 ──
idx = {'argilica': B['B11'] / (B['B12'] + eps), 'oxidos': B['B4'] / (B['B2'] + eps)}
M1 = {k: pct_mask(v, PERCENTIL) for k, v in idx.items()}

# ── M2: PCA dirigido ──
def pca_dirigido(bandas, pos, neg, pct):
    """PCA sobre `bandas`; elige la PC donde `pos` y `neg` tienen signos opuestos y mayor contraste."""
    X = np.stack([np.log(np.clip(B[b], 1e-4, None)) for b in bandas], -1)
    Xm = X[MASCARA]
    mu, sd = Xm.mean(0), Xm.std(0) + 1e-9
    n = min(len(Xm), 300000)
    sel = np.random.default_rng(0).choice(len(Xm), n, replace=False)
    p = PCA(n_components=len(bandas)).fit((Xm[sel] - mu) / sd)
    mejor, score = 0, -1
    for i, v in enumerate(p.components_):
        c = abs(v[bandas.index(pos)] - v[bandas.index(neg)])
        if v[bandas.index(pos)] * v[bandas.index(neg)] < 0 and c > score:
            mejor, score = i, c
    v = p.components_[mejor].copy()
    if v[bandas.index(pos)] < 0: v = -v                  # orientación: pos alto = más mineral
    pc = np.full((H, W), np.nan, 'float32')
    pc[MASCARA] = (((Xm - mu) / sd) @ v)
    pcv = np.where(np.isfinite(pc), pc, -9)
    return pcv, mejor + 1, v

pcA, kA, vA = pca_dirigido(['B2', 'B8', 'B11', 'B12'], pos='B11', neg='B12', pct=PERCENTIL)
pcO, kO, vO = pca_dirigido(['B2', 'B3', 'B4', 'B11'], pos='B4', neg='B2', pct=PERCENTIL)
M2 = {'argilica': pct_mask(pcA, PERCENTIL), 'oxidos': pct_mask(pcO, PERCENTIL)}
print(f'M2 arcillas → PC{kA} coef. {dict(zip(["B2","B8","B11","B12"], vA.round(2)))}')
print(f'M2 óxidos   → PC{kO} coef. {dict(zip(["B2","B3","B4","B11"], vO.round(2)))}')

# ── M3: SAM ──
X = np.moveaxis(R, 0, -1).reshape(-1, NB)
def sam_angulo(ref):
    num = X @ ref
    den = np.linalg.norm(X, axis=1) * np.linalg.norm(ref) + 1e-9
    return np.degrees(np.arccos(np.clip(num / den, -1, 1))).reshape(H, W)

def firma_desde_puntos(pts):
    sp = []
    for lon, lat in pts:
        x, y = transform_geom('EPSG:4326', CRS, {'type': 'Point', 'coordinates': (lon, lat)})['coordinates']
        c, r = ~TR * (x, y)
        if 0 <= int(r) < H and 0 <= int(c) < W and MASCARA[int(r), int(c)]:
            sp.append(R[:, int(r), int(c)])
    return np.mean(sp, 0) if sp else None

M3, firmas, origen = {}, {}, {}
for k in ['argilica', 'oxidos']:
    f = firma_desde_puntos(PUNTOS_REF.get(k, [])) if PUNTOS_REF.get(k) else None
    origen[k] = 'puntos del usuario' if f is not None else f'semilla automática (top {SEMILLA_PCT} % de M1)'
    if f is None:
        v = idx[k].copy(); v[~MASCARA] = -9
        semilla = v >= np.percentile(v[MASCARA], 100 - SEMILLA_PCT)
        f = R[:, semilla & MASCARA].mean(1)
    firmas[k] = f
    M3[k] = pct_mask(sam_angulo(f), PERCENTIL, alto=False)
print('M3 firma:', origen)

# ── Consenso ──
CONS, VOTOS = {}, {}
for k in ['argilica', 'oxidos']:
    VOTOS[k] = M1[k].astype(int) + M2[k].astype(int) + M3[k].astype(int)
    CONS[k] = VOTOS[k] >= 2
print(f'\n{"Clase":<22}{"1 método":>10}{"2 métodos":>11}{"3 métodos":>11}{"Consenso (≥2)":>15}{"% escena":>10}')
for nom, k in (('Arcillas / hidroxilos', 'argilica'), ('Óxidos de Fe', 'oxidos')):
    v = VOTOS[k] * MASCARA
    print(f'{nom:<22}' + ''.join(f'{(v == n_).sum()*KM2_PX:>{w_}.2f}' for n_, w_ in ((1, 10), (2, 11), (3, 11))) + f'{CONS[k].sum()*KM2_PX:>15.2f}{CONS[k].mean()*100:>9.1f}%   (km²)')
print('\nCada método marca por construcción el mismo % de la escena; lo informativo es cuánto COINCIDEN (columnas 2 y 3 métodos).')

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(14, 7.2))
for ax, (t, a) in zip(axs.ravel()[:2], [('Índice M1  B11/B12  (arcillas)', idx['argilica']), ('Índice M1  B4/B2  (óxidos Fe)', idx['oxidos'])]):
    v = np.where(MASCARA, a, np.nan)
    im = ax.imshow(v, extent=ext, cmap='magma', vmin=np.nanpercentile(v, 2), vmax=np.nanpercentile(v, 99))
    ax.set_title(t); plt.colorbar(im, ax=ax, shrink=.7)
for ax, (t, a) in zip(axs.ravel()[2:4], [(f'M2 PC{kA}  (arcillas)', pcA), (f'M2 PC{kO}  (óxidos)', pcO)]):
    v = np.where(MASCARA, a, np.nan)
    im = ax.imshow(v, extent=ext, cmap='viridis', vmin=np.nanpercentile(v, 2), vmax=np.nanpercentile(v, 99))
    ax.set_title(t); plt.colorbar(im, ax=ax, shrink=.7)
base = rgb(R) * 0.55
cmap_a = np.zeros((H, W, 4)); cmap_a[CONS['argilica']] = (0.95, 0.76, 0.19, 1)
cmap_o = np.zeros((H, W, 4)); cmap_o[CONS['oxidos']] = (0.88, 0.26, 0.24, 1)
ax = axs[1, 1]; ax.imshow(base, extent=ext); ax.imshow(cmap_a, extent=ext); ax.set_title('Consenso arcillas (≥ 2 de 3)')
ax = axs[1, 2]; ax.imshow(base, extent=ext); ax.imshow(cmap_o, extent=ext); ax.set_title('Consenso óxidos de Fe (≥ 2 de 3)')
axs[0, 2].axis('off')
axs[0, 2].text(0, .5, 'Cómo leer los mapas\n\n• M1/M2: más claro = más señal\n• Consenso: zonas donde coinciden\n  al menos 2 de los 3 métodos\n• Amarillo = arcillas (Al-OH)\n• Rojo = óxidos de Fe', fontsize=10, va='center')
for a in axs.ravel(): a.ticklabel_format(style='plain'); a.tick_params(labelsize=7)
plt.tight_layout(); plt.show()

## 4 · Lineamientos estructurales
Las fallas y fracturas controlan por dónde circulan los fluidos hidrotermales, así que su densidad es un criterio de prospectividad.

**Método:** imagen base → suavizado gaussiano → bordes **Canny** → segmentos **Hough** → se filtran por largo mínimo y se conservan los más largos. Con `USAR_DEM = True` la imagen base es un **relieve sombreado multi-azimut** de un DEM Copernicus 30 m (mucho más fiel a la estructura que la imagen óptica). Sin DEM se usa la primera componente principal de la escena (relieve + albedo).

In [ ]:
def hillshade(dem, az, alt=45, px=30):
    gy, gx = np.gradient(dem, px)
    slope = np.arctan(np.hypot(gx, gy)); asp = np.arctan2(-gx, gy)
    azr, altr = np.radians(az), np.radians(alt)
    hs = np.sin(altr) * np.cos(slope) + np.cos(altr) * np.sin(slope) * np.cos(azr - asp)
    return np.clip(hs, 0, 1)

def dem_a_malla(fuente):
    """Lleva cualquier raster de elevación (ruta o URL) a la malla de la escena."""
    dem = np.zeros((H, W), 'float32')
    with rasterio.open(fuente) as s:
        reproject(rasterio.band(s, 1), dem, dst_transform=TR, dst_crs=CRS, resampling=Resampling.bilinear)
    return dem

def descargar_dem():
    from pystac_client import Client
    cat = Client.open('https://earth-search.aws.element84.com/v1')
    l, b, r, t = transform_bounds(CRS, 'EPSG:4326', TR.c, TR.f + H * TR.e, TR.c + W * TR.a, TR.f)
    items = list(cat.search(collections=['cop-dem-glo-30'], bbox=[l, b, r, t]).items())
    if not items: raise RuntimeError('sin DEM para esta zona')
    mosaico = np.zeros((H, W), 'float32'); cnt = np.zeros((H, W), 'float32')
    with rasterio.Env(AWS_NO_SIGN_REQUEST='YES'):
        for it in items:
            d = dem_a_malla(it.assets['data'].href)
            ok = d != 0
            mosaico[ok] += d[ok]; cnt[ok] += 1
    return np.where(cnt > 0, mosaico / np.maximum(cnt, 1), np.nan)

# ── imagen base ──
DEM = None
if USAR_DEM:
    try:
        DEM = descargar_dem(); print('DEM Copernicus 30 m cargado')
    except Exception as e:
        print('⚠️ no se pudo descargar el DEM (', e, ') → se usa la imagen óptica')

if DEM is not None:
    d = np.where(np.isfinite(DEM), DEM, np.nanmean(DEM))
    d = gaussian(d, 1.0)
    base_lin = [hillshade(d, az, px=PX) for az in (0, 45, 90, 135)]
    fuente_lin = 'relieve sombreado multi-azimut (DEM)'
else:
    Xs = np.stack([np.log(np.clip(b, 1e-4, None)) for b in R], -1)
    Xm = Xs[MASCARA]
    p1 = PCA(1).fit((Xm - Xm.mean(0)) / (Xm.std(0) + 1e-9))
    pc1 = np.zeros((H, W), 'float32'); pc1[MASCARA] = ((Xm - Xm.mean(0)) / (Xm.std(0) + 1e-9)) @ p1.components_[0]
    lo, hi = np.percentile(pc1[MASCARA], (2, 98))
    base_lin = [np.clip((pc1 - lo) / (hi - lo), 0, 1)]
    fuente_lin = 'PC1 de la imagen óptica'
print('Imagen base:', fuente_lin)

# ── Canny + Hough ──
LMIN_PX = max(5, int(LONG_MIN_M / PX))
borde = np.zeros((H, W), bool)
for img in base_lin:
    borde |= canny(img, sigma=SIGMA_LIN, use_quantiles=True, low_threshold=Q_BAJO, high_threshold=Q_ALTO)   # umbrales por cuantiles: no dependen del contraste
segs = probabilistic_hough_line(borde, threshold=8, line_length=LMIN_PX, line_gap=6, rng=0)

LIN = []
for (x0, y0), (x1, y1) in segs:
    L = np.hypot(x1 - x0, y1 - y0) * PX
    if L < LONG_MIN_M: continue
    az = (np.degrees(np.arctan2(x1 - x0, -(y1 - y0)))) % 180      # azimut desde el norte, 0–180°
    LIN.append(dict(p0=(x0, y0), p1=(x1, y1), L=L, az=az))
LIN = sorted(LIN, key=lambda d: -d['L'])[:MAX_LIN]
print(f'Lineamientos: {len(LIN)} (largo medio {np.mean([l["L"] for l in LIN]):.0f} m · total {sum(l["L"] for l in LIN)/1000:.1f} km)' if LIN else 'No se detectaron lineamientos: baja LONG_MIN_M o SIGMA_LIN.')

# ── densidad de lineamientos (km de línea por km²) ──
rast = np.zeros((H, W), 'float32')
for l in LIN:
    rr, cc = draw_line(int(l['p0'][1]), int(l['p0'][0]), int(l['p1'][1]), int(l['p1'][0]))
    ok = (rr >= 0) & (rr < H) & (cc >= 0) & (cc < W)
    rast[rr[ok], cc[ok]] = 1
radio = max(1, int(500 / PX))
DENS_LIN = gaussian(rast, radio)            # densidad relativa de línea (0 = sin lineamientos)
DENS_LIN = np.where(MASCARA, DENS_LIN, 0)

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 5.2), gridspec_kw={'width_ratios': [1.35, 1, 1]})
ax = axs[0]
ax.imshow(rgb(R) * 0.7, extent=ext)
for l in LIN:
    (x0, y0), (x1, y1) = l['p0'], l['p1']
    ax.plot([TR.c + x0 * TR.a, TR.c + x1 * TR.a], [TR.f + y0 * TR.e, TR.f + y1 * TR.e], color='#22d3ee', lw=.9)
ax.set_title(f'Lineamientos ({len(LIN)})'); ax.ticklabel_format(style='plain'); ax.tick_params(labelsize=7)
im = axs[1].imshow(DENS_LIN, extent=ext, cmap='YlOrRd'); axs[1].set_title('Densidad de lineamientos (relativa)'); plt.colorbar(im, ax=axs[1], shrink=.7)
axs[1].ticklabel_format(style='plain'); axs[1].tick_params(labelsize=7)
axs[2].remove()
ax = fig.add_subplot(1, 3, 3, projection='polar')
if LIN:
    az = np.array([l['az'] for l in LIN]); w = np.array([l['L'] for l in LIN])
    bins = np.arange(0, 181, 10)
    h_, _ = np.histogram(az, bins=bins, weights=w)
    th = np.radians(np.r_[(bins[:-1] + 5), (bins[:-1] + 5) + 180])
    ax.bar(th, np.r_[h_, h_] / 1000, width=np.radians(10), color='#22d3ee', edgecolor='k', alpha=.85)
    ax.set_theta_zero_location('N'); ax.set_theta_direction(-1)
    dom = bins[np.argmax(h_)]
    ax.set_title(f'Rosa de lineamientos (km)\nDirección dominante N{dom}°–{dom+10}°', fontsize=10)
plt.tight_layout(); plt.show()

## 5 · Definición de zonas prioritarias
Se combinan dos criterios en un **puntaje de prospectividad (0–1)**:

`puntaje = peso_alteración × densidad de alteración + (1 − peso_alteración) × densidad de lineamientos`

* *Densidad de alteración*: consenso de arcillas **+** óxidos suavizado en un radio de `SUAVIZADO_M` (premia núcleos de alteración, no píxeles aislados). Donde coinciden ambas clases se suma más.
* *Zona Alta* = percentil superior del puntaje; se conserva solo lo que supera `AREA_MIN_KM2` y se ordena en un **ranking**.

In [ ]:
def norm01(a):
    v = a[MASCARA]
    hi = np.percentile(v, 99.5) if v.size else 1
    return np.clip(a / (hi + 1e-9), 0, 1)

sig = max(1, SUAVIZADO_M / PX)
alt = gaussian(CONS['argilica'].astype('float32') + CONS['oxidos'].astype('float32'), sig)
ALT = norm01(np.where(MASCARA, alt, 0))
LINN = norm01(DENS_LIN)
SCORE = np.where(MASCARA, PESO_ALTERACION * ALT + (1 - PESO_ALTERACION) * LINN, 0)

t_alta = np.percentile(SCORE[MASCARA], PERCENTIL_ALTA)
t_media = np.percentile(SCORE[MASCARA], 85)
CLASE = np.zeros((H, W), 'uint8'); CLASE[(SCORE >= t_media) & MASCARA] = 1; CLASE[(SCORE >= t_alta) & MASCARA] = 2

lab, n = ndi.label(CLASE == 2, structure=np.ones((3, 3)))
zonas = []
for i in range(1, n + 1):
    m = lab == i
    a = m.sum() * KM2_PX
    if a < AREA_MIN_KM2:
        lab[m] = 0; continue
    rr, cc = np.nonzero(m)
    x, y = TR * (cc.mean() + .5, rr.mean() + .5)
    lon, lat = transform_geom(CRS, 'EPSG:4326', {'type': 'Point', 'coordinates': (x, y)})['coordinates']
    lkm = 0.0
    for l in LIN:
        mx, my = int((l['p0'][0] + l['p1'][0]) / 2), int((l['p0'][1] + l['p1'][1]) / 2)
        if 0 <= my < H and 0 <= mx < W and m[my, mx]: lkm += l['L'] / 1000
    zonas.append(dict(old=i, area_km2=a, puntaje_medio=float(SCORE[m].mean()), puntaje_max=float(SCORE[m].max()),
                      pct_arcillas=100 * CONS['argilica'][m].mean(), pct_oxidos=100 * CONS['oxidos'][m].mean(),
                      lineamientos_km=lkm, lon=float(lon), lat=float(lat)))
zonas = sorted(zonas, key=lambda z: -(z['puntaje_medio'] * np.sqrt(z['area_km2'])))
remap = np.zeros(n + 1, 'int32')
for rank, z in enumerate(zonas, 1):
    z['zona'] = f'Z{rank:02d}'; remap[z['old']] = rank
LAB = remap[lab]

print(f'Zonas prioritarias: {len(zonas)}  ·  área total {sum(z["area_km2"] for z in zonas):.2f} km² ({100*sum(z["area_km2"] for z in zonas)/(W*H*KM2_PX):.1f} % de la escena)\n')
print(f'{"Zona":<6}{"Área km²":>9}{"Puntaje":>9}{"% arcillas":>12}{"% óxidos":>10}{"Lin. km":>9}   Lat / Lon')
for z in zonas[:15]:
    print(f'{z["zona"]:<6}{z["area_km2"]:>9.2f}{z["puntaje_medio"]:>9.2f}{z["pct_arcillas"]:>12.0f}{z["pct_oxidos"]:>10.0f}{z["lineamientos_km"]:>9.1f}   {z["lat"]:.4f}, {z["lon"]:.4f}')

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(14, 5.6))
im = axs[0].imshow(np.where(MASCARA, SCORE, np.nan), extent=ext, cmap='inferno'); axs[0].set_title('Puntaje de prospectividad (0–1)'); plt.colorbar(im, ax=axs[0], shrink=.75)
ax = axs[1]; ax.imshow(rgb(R) * 0.6, extent=ext)
cm = ListedColormap([(0, 0, 0, 0), (1, .75, .1, .45), (.9, .15, .15, .7)])
ax.imshow(CLASE, extent=ext, cmap=cm, vmin=0, vmax=2, interpolation='nearest')
for z in zonas[:12]:
    x, y = transform_geom('EPSG:4326', CRS, {'type': 'Point', 'coordinates': (z['lon'], z['lat'])})['coordinates']
    ax.text(x, y, z['zona'], color='white', fontsize=8, ha='center', weight='bold', bbox=dict(boxstyle='round,pad=.15', fc='black', alpha=.6, lw=0))
ax.set_title('Zonas: media (amarillo) y alta (rojo) con ranking')
for a in axs: a.ticklabel_format(style='plain'); a.tick_params(labelsize=7)
plt.tight_layout(); plt.show()

## 6 · (Opcional) Validación con tus muestras de campo
Si tienes puntos con mineralogía (por ejemplo del **TerraSpec**), pasa un CSV con columnas `lat`, `lon` (grados) y `mineral`. El cuaderno cuenta en qué porcentaje de puntos la imagen marcó la **misma clase** en un entorno de ±1 píxel.

> ⚠️ Con pocos puntos el resultado **no es concluyente** (intervalo de confianza muy ancho); recomendamos ≥ 30 por clase. Minerales que Sentinel-2 no ve (carbonatos, clorita, epidota) se excluyen.

In [ ]:
CSV_CAMPO = None        # ej. 'mis_muestras.csv'  (columnas: lat, lon, mineral)

CLASE_MINERAL = {
    'argilica': ['kaolin', 'dickit', 'alunit', 'pyrophyl', 'montmor', 'illit', 'muscov', 'white mica', 'smectit', 'sericit', 'halloys'],
    'oxidos':   ['jarosit', 'goethit', 'hematit', 'limonit', 'iron oxide'],
}
if CSV_CAMPO and os.path.exists(CSV_CAMPO):
    import csv as _csv
    filas = list(_csv.DictReader(open(CSV_CAMPO, encoding='utf-8-sig')))
    res = {'argilica': [0, 0], 'oxidos': [0, 0]}
    fuera = excl = 0
    for f in filas:
        m = f['mineral'].lower()
        k = next((c for c, ks in CLASE_MINERAL.items() if any(s in m for s in ks)), None)
        if not k: excl += 1; continue
        x, y = transform_geom('EPSG:4326', CRS, {'type': 'Point', 'coordinates': (float(f['lon']), float(f['lat']))})['coordinates']
        c, r = ~TR * (x, y); c, r = int(c), int(r)
        if not (1 <= r < H - 1 and 1 <= c < W - 1): fuera += 1; continue
        res[k][1] += 1
        res[k][0] += int(CONS[k][r - 1:r + 2, c - 1:c + 2].any())
    n_t = sum(v[1] for v in res.values())
    print(f'Puntos evaluados: {n_t}  (fuera de la imagen: {fuera}, mineral no detectable por S2: {excl})')
    for k, (ok, n_) in res.items():
        if n_: print(f'  {k:<9} {ok}/{n_} = {100*ok/n_:.0f} %   (azar ≈ {100*(1-(1-CONS[k].mean())**9):.0f} %)')
    if n_t < 30: print('⚠️ pocos puntos: resultado orientativo, no concluyente.')
else:
    print('Sin CSV de campo (CSV_CAMPO = None): se omite la validación.')

## 7 · Exportar resultados
Se guardan en la carpeta `resultados/` (en Colab: panel de archivos ◀ o la celda de descarga).

In [ ]:
def geom_ll(g): return transform_geom(CRS, 'EPSG:4326', g)

# GeoTIFF multibanda: puntaje, clase, consenso arcillas, consenso óxidos, densidad de lineamientos
with rasterio.open(f'{OUT}/prospectividad.tif', 'w', driver='GTiff', height=H, width=W, count=5, dtype='float32', crs=CRS, transform=TR, compress='deflate') as dst:
    for i, (a, d) in enumerate([(SCORE, 'puntaje'), (CLASE, 'clase_0baja_1media_2alta'), (CONS['argilica'], 'consenso_arcillas'), (CONS['oxidos'], 'consenso_oxidos'), (DENS_LIN, 'densidad_lineamientos')], 1):
        dst.write(a.astype('float32'), i); dst.set_band_description(i, d)

# Zonas → GeoJSON
datos = {z['zona']: z for z in zonas}
feats = []
for geom, val in features.shapes(LAB.astype('int32'), mask=LAB > 0, transform=TR):
    z = zonas[int(val) - 1]
    props = {k: (round(v, 3) if isinstance(v, float) else v) for k, v in z.items() if k != 'old'}
    feats.append({'type': 'Feature', 'properties': props, 'geometry': geom_ll(geom)})
json.dump({'type': 'FeatureCollection', 'features': feats}, open(f'{OUT}/zonas_prioritarias.geojson', 'w'))

# Lineamientos → GeoJSON
lf = []
for l in LIN:
    (x0, y0), (x1, y1) = l['p0'], l['p1']
    g = {'type': 'LineString', 'coordinates': [(TR.c + x0 * TR.a, TR.f + y0 * TR.e), (TR.c + x1 * TR.a, TR.f + y1 * TR.e)]}
    lf.append({'type': 'Feature', 'properties': {'largo_m': round(l['L']), 'azimut': round(l['az'])}, 'geometry': geom_ll(g)})
json.dump({'type': 'FeatureCollection', 'features': lf}, open(f'{OUT}/lineamientos.geojson', 'w'))

# Tabla CSV
cols = ['zona', 'area_km2', 'puntaje_medio', 'puntaje_max', 'pct_arcillas', 'pct_oxidos', 'lineamientos_km', 'lat', 'lon']
with open(f'{OUT}/zonas_prioritarias.csv', 'w', encoding='utf-8-sig') as f:
    f.write(','.join(cols) + '\n')
    for z in zonas: f.write(','.join(f'{z[c]:.4f}' if isinstance(z[c], float) else str(z[c]) for c in cols) + '\n')

# KMZ (Google Earth): zonas con etiqueta + lineamientos
def ring(coords): return ' '.join(f'{x:.6f},{y:.6f},0' for x, y in coords)
kml = ['<?xml version="1.0" encoding="UTF-8"?><kml xmlns="http://www.opengis.net/kml/2.2"><Document><name>Prospectividad</name>',
       '<Style id="z"><PolyStyle><color>990000ff</color></PolyStyle><LineStyle><color>ff0000ff</color><width>2</width></LineStyle></Style>',
       '<Style id="l"><LineStyle><color>ffd3d322</color><width>1</width></LineStyle></Style>',
       '<Style id="t"><IconStyle><scale>0</scale></IconStyle><LabelStyle><color>ffffffff</color><scale>1.1</scale></LabelStyle></Style>',
       '<Folder><name>Zonas prioritarias</name>']
for f in feats:
    p = f['properties']; g = f['geometry']
    polys = g['coordinates'] if g['type'] == 'MultiPolygon' else [g['coordinates']]
    desc = f"Área {p['area_km2']:.2f} km² · puntaje {p['puntaje_medio']:.2f} · arcillas {p['pct_arcillas']:.0f} % · óxidos {p['pct_oxidos']:.0f} % · lineamientos {p['lineamientos_km']:.1f} km"
    pm = ''.join(f'<Polygon><outerBoundaryIs><LinearRing><coordinates>{ring(po[0])}</coordinates></LinearRing></outerBoundaryIs></Polygon>' for po in polys)
    kml.append(f'<Placemark><name>{p["zona"]}</name><description>{desc}</description><styleUrl>#z</styleUrl><MultiGeometry>{pm}</MultiGeometry></Placemark>')
    kml.append(f'<Placemark><name>{p["zona"]}</name><styleUrl>#t</styleUrl><Point><coordinates>{p["lon"]:.6f},{p["lat"]:.6f},0</coordinates></Point></Placemark>')
kml.append('</Folder><Folder><name>Lineamientos</name><visibility>0</visibility>')
for f in lf:
    kml.append(f'<Placemark><styleUrl>#l</styleUrl><visibility>0</visibility><LineString><coordinates>{ring(f["geometry"]["coordinates"])}</coordinates></LineString></Placemark>')
kml.append('</Folder></Document></kml>')
with zipfile.ZipFile(f'{OUT}/prospectividad.kmz', 'w', zipfile.ZIP_DEFLATED) as z: z.writestr('doc.kml', ''.join(kml))

plt.imsave(f'{OUT}/mapa_zonas.png', np.clip(rgb(R) * 0.6 * (CLASE[..., None] == 0) + (CLASE[..., None] == 1) * np.array([1, .75, .1]) + (CLASE[..., None] == 2) * np.array([.9, .15, .15]), 0, 1))

with zipfile.ZipFile('prospectividad_resultados.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted(os.listdir(OUT)):
        if os.path.isfile(f'{OUT}/{f}') and not f.endswith('_descargada.tif') and f != EJEMPLO: z.write(f'{OUT}/{f}', f)
print('Archivos generados:'); [print(' ·', f'{OUT}/{f}', f'({os.path.getsize(f"{OUT}/{f}")/1024:.0f} KB)') for f in sorted(os.listdir(OUT))]
try:
    from google.colab import files
    files.download('prospectividad_resultados.zip')
except ImportError:
    pass

## 8 · Cómo interpretar y qué sigue
* **Las zonas «Alta» son hipótesis para visitar**, no blancos confirmados: la alteración detectada es **superficial** (Sentinel-2 ve unos pocos mm), puede ser supergénica o de cobertura, y la vegetación/suelos tapan áreas.
* **Mejoras con más datos:** SWIR fino (ASTER archivo, WorldView-3, EMIT, EnMAP, PRISMA) para separar argílica avanzada, fílica y propilítica · **firmas de campo** (TerraSpec/ASD) en `PUNTOS_REF` para M3 · **DEM** (`USAR_DEM = True`) para lineamientos · geoquímica y geofísica como capas extra del puntaje.
* **Reproducibilidad:** todos los umbrales son percentiles de *tu* escena; compara escenas solo con el mismo criterio y recuerda que cambian al variar el recorte.
